# Asset Class Momentum - Research Notebook

**Source**: [QC Investment Strategy Library - Asset Class Trend Following](https://www.quantconnect.com/learning/articles/investment-strategy-library/asset-class-trend-following)

**Concept**: Dual momentum across 5 asset class ETFs. Select the top 3 by 12-month momentum, weight equally, rebalance monthly.

In [1]:
try:
    from AlgorithmImports import *
    qb = QuantBook()
    QC_ENV = True
except (ImportError, NameError):
    import pandas as pd
    import numpy as np
    from datetime import timedelta
    QC_ENV = False
    qb = None

tickers = ["SPY", "EFA", "BND", "VNQ", "GSG"]

if QC_ENV:
    for t in tickers:
        qb.AddEquity(t, Resolution.Daily)
else:
    # Local research path: the same 5 liquid ETFs via yfinance, no degradation --
    # every asset class of the strategy is directly available.
    print("Local environment - loading the 5 asset-class ETFs via yfinance")

print(f"Universe: {tickers}")
print("Asset classes: US Equities, Intl Equities, Bonds, REITs, Commodities")

Local environment - loading the 5 asset-class ETFs via yfinance
Universe: ['SPY', 'EFA', 'BND', 'VNQ', 'GSG']
Asset classes: US Equities, Intl Equities, Bonds, REITs, Commodities


## Strategy Logic

1. Calculate 12-month (252-day) momentum for each ETF
2. Rank by momentum, select top 3
3. Equal weight (33.3% each)
4. Rebalance monthly

In [2]:
# Fetch historical data - QC Cloud or local yfinance, same frame downstream
if QC_ENV:
    history = qb.History(tickers, timedelta(252 * 12), Resolution.Daily)
    close = history["close"].unstack(level=0)
    print(f"Data shape: {history.shape}")
else:
    import yfinance as yf
    frames = {}
    for t in tickers:
        h = yf.Ticker(t).history(period="12y", auto_adjust=True)
        if not h.empty:
            h.index = h.index.tz_localize(None)  # normalize mixed tz (see env-gotchas)
            frames[t] = h["Close"]
    close = pd.DataFrame(frames).dropna()
    print(f"Close frame: {close.shape[0]} days x {close.shape[1]} ETFs")

close.tail()

Close frame: 3017 days x 5 ETFs


,SPY,EFA,BND,VNQ,GSG
Date,,,,,
2026-09-21,773.500000,106.129997,71.400002,93.003365,35.880001
2026-09-22,773.380005,106.500000,71.410004,92.815002,35.630001
2026-09-23,767.809998,104.800003,70.809998,91.449997,36.150002
2026-09-24,767.179993,104.500000,70.449997,91.169998,36.480000
2026-09-25,771.349976,105.559998,70.629997,90.989998,35.889999


In [3]:
# Calculate 12-month momentum for each asset (close frame built above, both paths)
momentum_252 = close.pct_change(252).iloc[-1]
print("Current 12-month momentum:")
for ticker, mom in momentum_252.sort_values(ascending=False).items():
    print(f"  {ticker}: {mom:.2%}")

top3 = momentum_252.sort_values(ascending=False).head(3).index.tolist()
print(f"\nTop 3 selection: {top3}")
print(f"Equal weight: {1/3:.1%} each")

Current 12-month momentum:
  GSG: 54.97%
  SPY: 17.94%
  EFA: 17.92%
  VNQ: 4.46%
  BND: -1.07%

Top 3 selection: ['GSG', 'SPY', 'EFA']
Equal weight: 33.3% each


## Key Observations

- **Trend following** is one of the most well-documented factor premia
- This strategy provides natural **diversification** across asset classes
- The 252-day lookback captures medium-term trends while filtering noise
- Monthly rebalancing keeps turnover manageable

### Strengths
- Simple, transparent, no ML dependency
- Captures momentum premium across multiple asset classes
- Low turnover = low transaction costs

### Risks
- Whipsaw in choppy/trendless markets
- All 3 selected assets may be correlated during crises
- 12-month lookback has significant lag

In [4]:
# Correlation matrix of the universe
returns = close.pct_change().dropna()
corr = returns.corr()
print("Correlation matrix (daily returns):")
print(corr.round(2))

Correlation matrix (daily returns):
      SPY   EFA   BND   VNQ   GSG
SPY  1.00  0.84  0.12  0.71  0.30
EFA  0.84  1.00  0.16  0.65  0.30
BND  0.12  0.16  1.00  0.29 -0.07
VNQ  0.71  0.65  0.29  1.00  0.18
GSG  0.30  0.30 -0.07  0.18  1.00


In [5]:
# REAL backtest: monthly top-3 by 12-month momentum, equal weight, vectorized
mom = close.pct_change(252)
rets = close.pct_change()

month_ends = close.resample("ME").last().index
months = [m for m in month_ends if mom.loc[:m].shape[0] > 252]

weights, strat = [], []
for i, m in enumerate(months[:-1]):
    signal = mom.loc[:m].iloc[-1].dropna()
    top3 = signal.sort_values(ascending=False).head(3).index
    nxt = rets.loc[(ret_idx := rets.index[rets.index > m])]
    nxt = nxt[nxt.index <= months[i + 1]]
    w = pd.Series(0.0, index=tickers)
    w[top3] = 1 / 3
    strat.append(nxt[w.index].mul(w, axis=1).sum(axis=1))

strat_ret = pd.concat(strat)
cum = (1 + strat_ret).cumprod()
years = (strat_ret.index[-1] - strat_ret.index[0]).days / 365.25
cagr = cum.iloc[-1] ** (1 / years) - 1
vol = strat_ret.std() * np.sqrt(252)
sharpe = cagr / vol if vol > 0 else float("nan")
dd = (cum / cum.cummax() - 1).min()

bench = (1 + rets["SPY"].reindex(strat_ret.index)).cumprod()
eqw5 = (1 + rets.reindex(strat_ret.index)[tickers].mean(axis=1)).cumprod()

print("=" * 62)
print("DUAL-MOMENTUM TOP-3 BACKTEST (12y, monthly rebalance, real data)")
print("=" * 62)
print(f"Months backtested: {len(months) - 1}")
print(f"Strategy CAGR: {cagr:.2%} | Vol: {vol:.2%} | Sharpe (rf=0): {sharpe:.2f} | MaxDD: {dd:.2%}")
print(f"SPY buy&hold CAGR: {bench.iloc[-1] ** (1 / years) - 1:.2%}")
print(f"Equal-weight 5 ETFs CAGR: {eqw5.iloc[-1] ** (1 / years) - 1:.2%}")
print(f"Final equity: strategy {cum.iloc[-1]:.2f}x | SPY {bench.iloc[-1]:.2f}x | EQW5 {eqw5.iloc[-1]:.2f}x")
print(f"\nCurrent selection (latest 252d momentum): {mom.iloc[-1].sort_values(ascending=False).head(3).index.tolist()}")
print("\nNote: no transaction costs, monthly turnover only; live monthly momentum vs this")
print("vectorized proxy: full engine (brokerage, slippage, live universes) on QC Cloud,")
print("companion strategy project.")

DUAL-MOMENTUM TOP-3 BACKTEST (12y, monthly rebalance, real data)
Months backtested: 132
Strategy CAGR: 10.25% | Vol: 13.45% | Sharpe (rf=0): 0.76 | MaxDD: -30.05%
SPY buy&hold CAGR: 15.38%
Equal-weight 5 ETFs CAGR: 8.51%
Final equity: strategy 2.92x | SPY 4.81x | EQW5 2.45x

Current selection (latest 252d momentum): ['GSG', 'SPY', 'EFA']

Note: no transaction costs, monthly turnover only; live monthly momentum vs this
vectorized proxy: full engine (brokerage, slippage, live universes) on QC Cloud,
companion strategy project.


## Reading the backtest

The numbers above are **measured on real daily closes**, not "expected
characteristics": the monthly top-3 selection is simulated end-to-end over
the available history with a 252-day lookback and next-month holding. Compare
the strategy line against the two honest references computed on the same
window -- SPY buy & hold and the naive equal-weight five -- before drawing
any conclusion. No costs or slippage are modeled here; the production engine
on QC Cloud remains the reference implementation.